# Week 4 Example Solution: Predict & Explain — Regression on Real DASS Data

This notebook demonstrates a complete regression pipeline: loading the OpenPsychometrics DASS-42 dataset, scoring the DASS Depression subscale and Big Five personality traits, building and comparing regression models, and interpreting the results.

**Key findings:**
- R² ≈ 0.34 — personality and demographics explain about a third of the variance in depression scores
- Emotional Stability is the strongest single predictor (coefficient ≈ −3.4)
- All three models (Linear, Ridge, Lasso) perform nearly identically with this much data
- No overfitting — training and test performance are very similar

---

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso, lasso_path
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded successfully!")

## 1. Load and Clean the Data

The OpenPsychometrics DASS-42 dataset is tab-separated (not comma-separated). It has 39,775 respondents and 172 columns. We need to:
1. Filter unreasonable ages (some look like birth years)
2. Remove careless responders using the vocabulary check fake words

In [ ]:
# Load the data (tab-separated!)
data = pd.read_csv("../data/dass42_data.csv", sep="\t")
print(f"Raw dataset: {data.shape[0]} rows × {data.shape[1]} columns")

# Filter unreasonable ages
data = data[(data["age"] >= 10) & (data["age"] <= 100)]
print(f"After age filter (10-100): {len(data)} rows")

# Remove careless responders (endorsed 2+ fake vocabulary words)
fake_words = ["VCL6", "VCL9", "VCL12"]
data["vcl_fake_count"] = data[fake_words].sum(axis=1)
n_careless = (data["vcl_fake_count"] >= 2).sum()
data = data[data["vcl_fake_count"] < 2]
print(f"Removed {n_careless} careless responders")
print(f"Clean dataset: {len(data)} rows")

## 2. Score the DASS Depression Subscale

The DASS-42 Depression subscale consists of 14 items: Q3, Q5, Q10, Q13, Q16, Q17, Q21, Q24, Q26, Q31, Q34, Q37, Q38, Q42.

The raw items are coded **1–4** in this dataset, but standard DASS scoring uses **0–3**. We subtract 1 from each item, then sum them. The total ranges from 0 to 42.

In [ ]:
# Score DASS Depression (14 items, recode 1-4 → 0-3, sum)
dep_items = [f"Q{i}A" for i in [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]]
data["DASS_Depression"] = data[dep_items].sub(1).sum(axis=1)

# Also score Anxiety and Stress for reference
anx_items = [f"Q{i}A" for i in [2, 4, 7, 9, 15, 19, 20, 23, 25, 29, 30, 33, 36, 40]]
stress_items = [f"Q{i}A" for i in [1, 6, 8, 11, 12, 14, 18, 22, 27, 28, 32, 35, 39, 41]]
data["DASS_Anxiety"] = data[anx_items].sub(1).sum(axis=1)
data["DASS_Stress"] = data[stress_items].sub(1).sum(axis=1)

print("DASS Subscale Statistics:")
for subscale in ["DASS_Depression", "DASS_Anxiety", "DASS_Stress"]:
    print(f"  {subscale}: M={data[subscale].mean():.1f}, SD={data[subscale].std():.1f}, "
          f"Median={data[subscale].median():.0f}, Range=[{data[subscale].min():.0f}, {data[subscale].max():.0f}]")

**DASS Depression: M = 21.0, SD = 12.3.** This is a high mean — the clinical cut-off for "severe" depression on the DASS-42 is 21+. This reflects the self-selected online sample: people with more symptoms are more likely to seek out a depression questionnaire.

## 3. Score the Big Five Personality Traits (TIPI)

The Ten-Item Personality Inventory measures five personality traits using just 10 items. Items 2, 4, 6, 8, and 10 are reverse-scored (reverse = 8 − original). Each trait is the average of its two items.

In [ ]:
# Reverse-score TIPI items 2, 4, 6, 8, 10
for i in [2, 4, 6, 8, 10]:
    data[f"TIPI{i}R"] = 8 - data[f"TIPI{i}"]

# Compute Big Five scales
data["Extraversion"] = (data["TIPI1"] + data["TIPI6R"]) / 2
data["Agreeableness"] = (data["TIPI2R"] + data["TIPI7"]) / 2
data["Conscientiousness"] = (data["TIPI3"] + data["TIPI8R"]) / 2
data["EmotionalStability"] = (data["TIPI4R"] + data["TIPI9"]) / 2
data["Openness"] = (data["TIPI5"] + data["TIPI10R"]) / 2

big5 = ["Extraversion", "Agreeableness", "Conscientiousness", "EmotionalStability", "Openness"]
print("Big Five Statistics:")
for trait in big5:
    print(f"  {trait}: M={data[trait].mean():.2f}, SD={data[trait].std():.2f}")

## 4. Explore Correlations

Before modelling, let's see which features are most correlated with DASS Depression. This guides our feature selection.

In [ ]:
# Correlations with DASS Depression
all_predictors = big5 + ["age", "gender", "education", "urban", "married", "familysize"]
corrs = data[["DASS_Depression"] + all_predictors].corr()["DASS_Depression"].drop("DASS_Depression")
corrs_sorted = corrs.reindex(corrs.abs().sort_values(ascending=False).index)

print("Correlations with DASS Depression (sorted by strength):")
for var in corrs_sorted.index:
    print(f"  {var:25s}  r = {corrs_sorted[var]:+.3f}")

**Key finding:** Emotional Stability has the strongest correlation with depression (r = −0.52). This makes sense — Emotional Stability is the inverse of neuroticism, which has been consistently linked to depression in personality research for decades.

Extraversion (r = −0.29) and Conscientiousness (r = −0.29) are moderate predictors. Demographics are weaker.

## 5. Feature Selection and Train/Test Split

In [ ]:
# Features: Big Five + demographics
features = big5 + ["age", "gender", "education", "urban", "married", "familysize"]
target = "DASS_Depression"

# Drop rows with missing values
model_data = data[features + [target]].dropna()
X = model_data[features]
y = model_data[target]

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Features: {features}")
print(f"Total samples: {len(X)}")
print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)} (locked until final evaluation!)")

## 6. Baseline Model

The simplest possible prediction: guess the mean depression score for everyone. This gives us a floor — any real model should beat this.

In [ ]:
# Baseline: predict the mean
baseline = DummyRegressor(strategy="mean")
baseline_cv_mae = -cross_val_score(baseline, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
baseline_cv_r2 = cross_val_score(baseline, X_train, y_train, cv=5, scoring="r2")
baseline.fit(X_train, y_train)

print(f"Baseline (mean prediction):")
print(f"  Predicted value for everyone: {y_train.mean():.2f}")
print(f"  CV MAE: {baseline_cv_mae.mean():.2f} ± {baseline_cv_mae.std():.2f}")
print(f"  CV R²:  {baseline_cv_r2.mean():.4f}")
print(f"\nThis is the number to beat: MAE = {baseline_cv_mae.mean():.2f}")

**Baseline MAE ≈ 10.59.** On average, predicting the mean is off by about 10.6 points on a 0–42 scale. Can our models do better?

## 7. Build and Cross-Validate Models

In [ ]:
# Build three regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso (alpha=0.1)": Lasso(alpha=0.1, max_iter=10000),
}

# Store results for comparison
results = {
    "Model": ["Baseline (mean)"],
    "CV MAE": [baseline_cv_mae.mean()],
    "CV R²": [baseline_cv_r2.mean()],
    "Train R²": [0.0],
}

for name, model in models.items():
    cv_mae = -cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    model.fit(X_train, y_train)
    train_r2 = model.score(X_train, y_train)

    results["Model"].append(name)
    results["CV MAE"].append(cv_mae.mean())
    results["CV R²"].append(cv_r2.mean())
    results["Train R²"].append(train_r2)

    print(f"{name}:")
    print(f"  CV MAE: {cv_mae.mean():.2f} ± {cv_mae.std():.2f}")
    print(f"  CV R²:  {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
    print(f"  Train R²: {train_r2:.4f}")
    print()

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**All three models perform nearly identically!** CV MAE ≈ 8.19, CV R² ≈ 0.339. With 30,896 training rows and only 11 features, there's no overfitting to regularise away. This is an important lesson: **regularisation matters most when you have many features relative to observations**. With this much data, ordinary least squares works just fine.

Also note: Train R² (0.340) and CV R² (0.339) are virtually identical — confirming no overfitting.

## 8. Model Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colours = ["#8C8C8C", "#4A90D9", "#5BA55B", "#E8873D"]

# MAE comparison
axes[0].bar(results_df["Model"], results_df["CV MAE"], color=colours, edgecolor="white", linewidth=0.5)
axes[0].set_ylabel("Cross-Validated MAE", fontsize=12)
axes[0].set_title("Model Comparison: Mean Absolute Error", fontsize=14, fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(results_df["CV MAE"]):
    axes[0].text(i, v + 0.1, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")

# R² comparison
axes[1].bar(results_df["Model"], results_df["CV R²"], color=colours, edgecolor="white", linewidth=0.5)
axes[1].set_ylabel("Cross-Validated R²", fontsize=12)
axes[1].set_title("Model Comparison: R² (Variance Explained)", fontsize=14, fontweight="bold")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(results_df["CV R²"]):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
fig.savefig("images/model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: images/model_comparison.png")

## 9. Coefficient Analysis

The coefficients from linear regression tell us how much each feature contributes to the prediction. A coefficient of −3.4 for Emotional Stability means: for every 1-point increase in Emotional Stability, the model predicts a 3.4-point decrease in Depression score.

In [ ]:
# Get coefficients from linear regression
lr_model = models["Linear Regression"]
coefs = pd.Series(lr_model.coef_, index=features)
coefs_sorted = coefs.sort_values()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bar_colours = ["#A71930" if c < 0 else "#4A90D9" for c in coefs_sorted]
coefs_sorted.plot(kind="barh", ax=ax, color=bar_colours, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Coefficient Value", fontsize=12)
ax.set_title("Regression Coefficients: What Predicts Depression?", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.8)

# Add value labels
for i, (val, name) in enumerate(zip(coefs_sorted, coefs_sorted.index)):
    offset = -0.15 if val < 0 else 0.05
    ax.text(val + offset, i, f"{val:.2f}", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig("images/coefficients.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: images/coefficients.png")

print("\nCoefficients (sorted by magnitude):")
for feat, coef in sorted(zip(features, lr_model.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:25s} {coef:+.3f}")

**Interpretation:**

- **Emotional Stability (−3.40)** is by far the strongest predictor. This is the inverse of neuroticism — people who are more emotionally stable report much lower depression. This aligns perfectly with decades of personality–psychopathology research.
- **Extraversion (−1.42)** is the second strongest — more extraverted people tend to be less depressed. This is consistent with the idea that social engagement is protective.
- **Conscientiousness (−1.07)** — more organised, disciplined people report lower depression.
- **Demographics add relatively little** beyond personality. Married status (−0.78) and education (−0.59) have modest effects; age, gender, urban/rural, and family size are near zero.

The Lasso model tells a similar story — at alpha=0.5, it drops all demographic features but keeps all five personality traits.

## 10. Final Test Set Evaluation

In [ ]:
# Evaluate best model on held-out test set
best_model = models["Linear Regression"]
y_pred = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

cv_r2 = results_df.loc[results_df["Model"] == "Linear Regression", "CV R²"].values[0]

print(f"Final Test Set Evaluation (Linear Regression):")
print(f"  Test MAE: {test_mae:.2f}")
print(f"  Test R²:  {test_r2:.4f}")
print(f"  CV R²:    {cv_r2:.4f}")
print(f"  Gap:      {abs(cv_r2 - test_r2):.4f}")
print(f"\n→ Minimal gap between CV and test = no overfitting!")

## 11. Residual Plots

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_pred, y_test, alpha=0.1, s=5, color="#4A90D9")
axes[0].plot([0, 42], [0, 42], "r--", linewidth=2, label="Perfect prediction")
axes[0].set_xlabel("Predicted Depression Score", fontsize=12)
axes[0].set_ylabel("Actual Depression Score", fontsize=12)
axes[0].set_title("Predicted vs Actual", fontsize=14, fontweight="bold")
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.1, s=5, color="#E8873D")
axes[1].axhline(y=0, color="red", linewidth=2, linestyle="--")
axes[1].set_xlabel("Predicted Depression Score", fontsize=12)
axes[1].set_ylabel("Residual (Actual - Predicted)", fontsize=12)
axes[1].set_title("Residuals vs Predicted", fontsize=14, fontweight="bold")

plt.tight_layout()
fig.savefig("images/residuals.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: images/residuals.png")

**Residual analysis:** The predicted vs actual plot shows points clustered around the diagonal, with substantial spread. The residuals plot shows roughly symmetric errors centred at zero, though there's a slight fan shape — the model has more variance in its errors at the extremes. This suggests the linear model captures the main trend but misses some non-linear relationships.

## Bonus: Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    LinearRegression(), X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring="neg_mean_absolute_error",
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, -train_scores.mean(axis=1), "o-", color="#4A90D9", label="Training MAE")
ax.fill_between(train_sizes,
                -train_scores.mean(axis=1) - train_scores.std(axis=1),
                -train_scores.mean(axis=1) + train_scores.std(axis=1),
                alpha=0.1, color="#4A90D9")
ax.plot(train_sizes, -val_scores.mean(axis=1), "o-", color="#A71930", label="Validation MAE")
ax.fill_between(train_sizes,
                -val_scores.mean(axis=1) - val_scores.std(axis=1),
                -val_scores.mean(axis=1) + val_scores.std(axis=1),
                alpha=0.1, color="#A71930")
ax.set_xlabel("Training Set Size", fontsize=12)
ax.set_ylabel("Mean Absolute Error", fontsize=12)
ax.set_title("Learning Curves: Linear Regression", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig("images/learning_curves.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: images/learning_curves.png")

**Learning curves:** Training and validation errors converge quickly and remain close throughout. This confirms there's no overfitting — we have more than enough data for 11 features. Adding more data wouldn't help; the model's limitation is the features themselves (personality and demographics can only explain so much about depression).

## Bonus: Lasso Regularisation Path

In [ ]:
alphas_lasso, coefs_path, _ = lasso_path(X_train, y_train, alphas=np.logspace(-2, 1, 50))

fig, ax = plt.subplots(figsize=(10, 6))
feature_colours = ["#4A90D9", "#5BA55B", "#E8873D", "#A71930", "#7B68A8",
                    "#8C8C8C", "#C8972C", "#2D2D2D", "#D4533B", "#4A90D9", "#5BA55B"]
for i, feat in enumerate(features):
    ax.plot(alphas_lasso, coefs_path[i], label=feat, color=feature_colours[i], linewidth=2)

ax.set_xscale("log")
ax.set_xlabel("Lasso Alpha (regularisation strength)", fontsize=12)
ax.set_ylabel("Coefficient Value", fontsize=12)
ax.set_title("Lasso Path: How Coefficients Change with Regularisation", fontsize=14, fontweight="bold")
ax.axhline(y=0, color="black", linewidth=0.5, linestyle=":")
ax.legend(fontsize=8, loc="upper right", ncol=2)
plt.tight_layout()
fig.savefig("images/lasso_path.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: images/lasso_path.png")

**Lasso path:** As regularisation increases (moving right), Lasso progressively zeros out features:
- First to go: gender, married — weakest predictors
- Then: education, urban, familysize — all demographics
- Agreeableness and Openness drop next
- **Emotional Stability is the last feature standing** — it survives even strong regularisation

This tells us that if we could only use one feature to predict depression, Emotional Stability (or equivalently, neuroticism) would be the best choice. This is exactly what personality psychology predicts.

## Summary

| Finding | Value |
|---------|-------|
| Clean dataset | 38,620 respondents |
| Best model | Linear Regression (all models tied) |
| CV MAE | 8.19 (baseline: 10.59) |
| CV R² | 0.339 |
| Test R² | 0.334 |
| Overfitting gap | 0.006 (negligible) |
| Top predictor | Emotional Stability (coef = −3.40) |

**Key takeaways:**
1. Personality explains about a third of depression variance — strong for survey data
2. Emotional Stability (inverse of neuroticism) dominates, as decades of research predicts
3. With 38K rows and 11 features, regularisation is unnecessary — the models are identical
4. Real data is messier than synthetic data but produces more psychologically meaningful results